In [167]:
import pandas as pd
import numpy as np
import sqlite3

In [168]:
inventory_levels = pd.read_csv('INVENTORY_LEVELS-data.csv')
product_forecast = pd.read_csv('PRODUCT_FORECAST-data.csv')
sales_target = pd.read_csv('SALES_TARGET-data.csv', encoding='cp1252')

crm = sqlite3.connect('CRM-data.sqlite')
go_sales = sqlite3.connect('GO_SALES-data.sqlite')
go_staff = sqlite3.connect('GO_STAFF-data.sqlite')


In [169]:
age_group = pd.read_sql("SELECT * FROM age_group;",crm)
customer_contact = pd.read_sql("SELECT * FROM customer_contact;",crm)
customer_segment = pd.read_sql("SELECT * FROM customer_segment;",crm)
customer_type = pd.read_sql("SELECT * FROM customer_type;",crm)
sales_territory = pd.read_sql("SELECT * FROM sales_territory;",crm)
customer_store = pd.read_sql("SELECT * FROM customer_store;",crm)
crm_country = pd.read_sql("SELECT * FROM crm_country;",crm)
customer = pd.read_sql("SELECT * FROM customer;",crm)
sales_demographic = pd.read_sql("SELECT * FROM sales_demographic;",crm)
customer_headquarters = pd.read_sql("SELECT * FROM customer_headquarters;",crm)

order_method = pd.read_sql("SELECT * FROM order_method;",go_sales)
product_line = pd.read_sql("SELECT * FROM product_line;",go_sales)
retailer_site = pd.read_sql("SELECT * FROM retailer_site;",go_sales)
return_reason = pd.read_sql("SELECT * FROM return_reason;",go_sales)
returned_item = pd.read_sql("SELECT * FROM returned_item;",go_sales)
order_details = pd.read_sql("SELECT * FROM order_details;",go_sales)
order_header = pd.read_sql("SELECT * FROM order_header;",go_sales)
product_type = pd.read_sql("SELECT * FROM product_type;",go_sales)
product = pd.read_sql("SELECT * FROM product;",go_sales)
sales_staff = pd.read_sql("SELECT * FROM sales_staff;",go_sales)
country = pd.read_sql("SELECT * FROM country;",go_sales)
sales_branch = pd.read_sql("SELECT * FROM sales_branch;",go_sales)

satisfaction = pd.read_sql("SELECT * FROM satisfaction;",go_staff)
training = pd.read_sql("SELECT * FROM training;",go_staff)
course = pd.read_sql("SELECT * FROM course;",go_staff)
sales_office = pd.read_sql("SELECT * FROM sales_office;",go_staff)
satisfaction_type = pd.read_sql("SELECT * FROM satisfaction_type;",go_staff)
sales_representative = pd.read_sql("SELECT * FROM sales_representative;",go_staff)

In [170]:
# ik maak er een lijst van om in 1 overzicht te kunnen zien in welke databases een lege colom zit

lijstje = [
    ("inventory_levels", inventory_levels),
    ("product_forecast", product_forecast),
    ("sales_target", sales_target),
    ("age_group", age_group),
    ("customer_contact", customer_contact),
    ("customer_segment", customer_segment),
    ("customer_type", customer_type),
    ("sales_territory", sales_territory),
    ("customer_store", customer_store),
    ("crm_country", crm_country),
    ("customer", customer),
    ("sales_demographic", sales_demographic),
    ("customer_headquarters", customer_headquarters),
    ("order_method", order_method),
    ("product_line", product_line),
    ("retailer_site", retailer_site),
    ("return_reason", return_reason),
    ("order_details", order_details),
    ("order_header", order_header),
    ("product_type", product_type),
    ("product", product),
    ("sales_staff", sales_staff),
    ("country", country),
    ("sales_branch", sales_branch),
    ("satisfaction", satisfaction),
    ("training", training),
    ("course", course),
    ("sales_office", sales_office),
    ("satisfaction_type", satisfaction_type),
    ("sales_representative", sales_representative)
]


for naam, df in lijstje:
    for col in df.columns:
        if df[col].isna().any():
            print(f"{naam} - {col}")

customer_contact - EXTENSION
customer_contact - FAX
customer_contact - E_MAIL
customer_store - ADDITION
customer_store - STATE
customer_store - ZIPCODE
customer - CUSTOMER_CODEMR
customer_headquarters - ADDRESS2
customer_headquarters - REGION
customer_headquarters - POSTAL_ZONE
customer_headquarters - PHONE
customer_headquarters - FAX
retailer_site - ADDRESS2
retailer_site - REGION
retailer_site - POSTAL_ZONE
sales_staff - EXTENSION
sales_branch - ADDRESS2
sales_branch - REGION
sales_office - ADDITION
sales_office - REGION
sales_representative - EXTENSION


In [171]:
sales_representative['EXTENSION'] = sales_representative['EXTENSION'].fillna(0)

sales_office['ADDITION'] = sales_office['ADDITION'].fillna('-')
sales_office['REGION'] = sales_office['REGION'].fillna(sales_office['REGION'].mode()[0])

sales_branch['ADDRESS2'] = sales_branch['ADDRESS2'].fillna(sales_branch['ADDRESS1'].mode()[0])
sales_branch['REGION'] = sales_branch['REGION'].fillna(sales_branch['REGION'].mode()[0])

sales_staff['EXTENSION'] = sales_staff['EXTENSION'].fillna(sales_staff['EXTENSION'].mode()[0])

smollist = 'REGION', 'POSTAL_ZONE'
retailer_site['ADDRESS2'] = retailer_site['ADDRESS2'].fillna(retailer_site['ADDRESS1'])
for i in smollist:
    retailer_site[i] = retailer_site[i].fillna(retailer_site[i].mode()[0])

smollist = 'REGION', 'POSTAL_ZONE', 'PHONE', 'FAX'
customer_headquarters['ADDRESS2'] = customer_headquarters['ADDRESS2'].fillna(customer_headquarters['ADDRESS1'])
for i in smollist:
    customer_headquarters[i] = customer_headquarters[i].fillna(customer_headquarters[i].mode()[0])

customer['CUSTOMER_CODEMR'] = customer['CUSTOMER_CODEMR'].fillna(customer['CUSTOMER_CODEMR'].mode()[0])

smollist = 'STATE', 'ZIPCODE'

customer_store['ADDITION'] = customer_store['ADDITION'].fillna('-')
customer_store['STATE']
for i in smollist:
    customer_store[i] = customer_store[i].fillna(customer_store[i].mode()[0])

smollist = 'FAX', 'E_MAIL'
customer_contact['EXTENSION'] = customer_contact['EXTENSION'].fillna('0')
for i in smollist:
    customer_contact[i] = customer_contact[i].fillna(customer_contact[i].mode()[0])

product_forecast['PRODUCT_FORECAST_CODE'] = product_forecast.index
sales_target['SALES_TARGET_CODE'] = sales_target.index
inventory_levels['INVENTORY_LEVELS_CODE'] = inventory_levels.index
satisfaction['SATISFACTION_CODE'] = satisfaction.index
training['TRAINING_CODE'] = training.index

In [172]:
sales_representative = sales_representative.drop(columns='EXTENSION')

DIM_Sales_Staff = pd.merge(
    sales_representative,
    sales_staff,
    how='outer',
    on=['FIRST_NAME', 'LAST_NAME', 'POSITION_EN',
       'WORK_PHONE', 'FAX', 'EMAIL', 'DATE_HIRED']
)

DIM_Sales_Staff.insert(0, 'SALES_STAFF_CODE', DIM_Sales_Staff.pop('SALES_STAFF_CODE'))


In [173]:
crm_country = crm_country.rename(columns={'COUNTRY_EN':'COUNTRY'})

DIM_Country = pd.merge(
    country,
    crm_country,
    how='outer',
    on=['COUNTRY_CODE', 'COUNTRY']
)
DIM_Country = pd.merge(
    DIM_Country,
    sales_territory,
    how='left',
    on=['SALES_TERRITORY_CODE']
)


In [174]:
sales_office = sales_office.rename(columns={'ZIPCODE':'POSTAL_ZONE',
                                            'COUNTRY': 'COUNTRY_CODE'})

DIM_Sales_Branch = pd.merge(
    sales_office,
    sales_branch,
    how='outer',
    on=['CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY_CODE']
)

In [175]:
customer_store = customer_store.rename(columns={'ZIPCODE':'POSTAL_ZONE',
                                                'STATE': 'REGION'})

DIM_Retailer_Site = pd.merge(
    retailer_site,
    customer_store,
    how='outer',
    on=['CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY_CODE', 'ACTIVE_INDICATOR']
)

In [176]:
customer = customer.merge(customer_type, how='left', on='CUSTOMER_TYPE_CODE')
customer = customer.rename(columns={'CUSTOMER_CODEMR':'CUSTOMER_HEADQUARTERS'})

In [177]:
returned_item = returned_item.merge(return_reason,on='RETURN_REASON_CODE',how='left')

In [178]:
Month = pd.DataFrame()
Date = pd.DataFrame()

In [179]:
returned_item['RETURN_DATE'] = pd.to_datetime(returned_item['RETURN_DATE'], format="%d-%b-%Y %I:%M:%S %p")
product['INTRODUCTION_DATE'] = pd.to_datetime(product['INTRODUCTION_DATE'], format="%d-%b-%Y %I:%M:%S %p")
order_header['ORDER_DATE'] = pd.to_datetime(order_header['ORDER_DATE'], format="%d-%b-%Y %I:%M:%S %p")
DIM_Sales_Staff['DATE_HIRED'] = pd.to_datetime(DIM_Sales_Staff['DATE_HIRED'], format="%d-%b-%Y %I:%M:%S %p")

In [180]:
Month = pd.concat([
    pd.DataFrame({
        'YEAR': inventory_levels['INVENTORY_YEAR'],
        'MONTHNR': inventory_levels['INVENTORY_MONTH']
    }),
    
    pd.DataFrame({
        'YEAR': product_forecast['YEAR'],
        'MONTHNR': product_forecast['MONTH']
    })
]).drop_duplicates().sort_values(['YEAR', 'MONTHNR']).reset_index(drop=True)

Month['MONTH'] = Month['MONTHNR'].map({1 : 'Jan', 2 : 'Feb', 3 : 'Mar', 4 : 'Apr', 5:'Mei', 6 : 'Jun', 7 : 'Jul', 8 : 'Aug', 9 : 'Sep', 10 : 'Oct', 11 : 'Nov', 12 : 'Dec'})
Month['QUARTER'] = Month['MONTHNR'].map({ 2 : 1, 3 : 1, 4 : 2, 5:2, 6 : 2, 7 : 3, 8 : 3, 9 : 3, 10 : 4, 11 : 4, 12 : 4})
Month['MONTH_ID'] = Month.index

In [181]:
Date['DATUM'] = pd.concat([
    returned_item['RETURN_DATE'],
    product['INTRODUCTION_DATE'],
    order_header['ORDER_DATE'],
    DIM_Sales_Staff['DATE_HIRED']
]).drop_duplicates().sort_values().reset_index(drop=True)

Date['DAY'] = pd.to_datetime(Date['DATUM']).dt.day

Date['YEAR'] = Date['DATUM'].dt.year
Date['MONTHNR'] = Date['DATUM'].dt.month

Date = Date.merge(
    Month[['YEAR', 'MONTHNR', 'MONTH_ID']],
    on=['YEAR', 'MONTHNR'],
    how='left'
)
Date = Date.rename(columns={'MONTHNR' : 'MONTH'})
Date['DATUM'] = Date['DATUM'].astype(str)
Date['DATE_ID'] = Date.index

In [182]:
order_header = order_header.merge(order_method, how='left', on='ORDER_METHOD_CODE')
order_header = order_header.rename(columns={'SALES_STAFF_CODE':'SALES_STAFF', 'RETAILER_SITE_CODE':'RETAILER_SITE', 'SALES_BRANCH_CODE':'SALES_BRANCH'})
order_header['ORDER_DATE'] = order_header['ORDER_DATE'].astype(str)

In [183]:
returned_item = returned_item.rename(columns={'ORDER_DETAIL_CODE':'ORDER_DETAILS', 'RETURN_QUANTITY':'QUANTITY_CATEGORY', 'RETURN_DESCRIPTION_EN':'RETURN_REASON_EN'})
returned_item['RETURN_DATE'] = returned_item['RETURN_DATE'].astype(str)

In [184]:
returned_item

,RETURN_CODE,RETURN_DATE,ORDER_DETAILS,RETURN_REASON_CODE,QUANTITY_CATEGORY,RETURN_REASON_EN
0,1491,2024-08-01 04:10:24,84858,5,8,Unsatisfactory product
1,1492,2023-12-06 18:46:19,84440,2,2,Incomplete product
2,1493,2024-06-24 09:23:14,84867,4,22,Wrong product shipped
3,1494,2024-07-21 00:00:09,84873,3,20,Wrong product ordered
4,1496,2023-10-07 05:13:58,84488,1,2,Defective product
...,...,...,...,...,...,...
685,2465,2024-06-06 09:22:12,114972,3,24,Wrong product ordered
686,2466,2024-10-06 12:31:25,114983,5,66,Unsatisfactory product
687,2467,2025-03-31 15:39:38,115215,4,64,Wrong product shipped
688,2468,2025-12-23 19:48:50,115171,1,2,Defective product


In [185]:
order_details['UNIT_PRICE'] = order_details['UNIT_PRICE'].astype('double')
order_details['UNIT_COST'] = order_details['UNIT_COST'].astype('double')

order_details['UNIT_REVENUE'] = (
    order_details['UNIT_PRICE'] - order_details['UNIT_COST']
)

In [186]:
# ✔ DIM_age_group
# ✔ DIM_customer_segment
# ✔ DIM_Month
# ✔ DIM_country
# ✔ DIM_Date
# ✔ DIM_sales_branch
# ✔ DIM_product
# ✔ DIM_customer_headquarters
# ❌ DIM_sales_staff mist kolommen: {'TRAINING_COURSE', 'MANAGER', 'STATISFACTION_YEAR', 'TRAINING_YEAR', 'STATISFACTION_DESCRIPTION'}
# ✔ DIM_customer
# ✔ DIM_retailer_site
# ✔ FACT_sales_demographic
# ✔ FACT_order_header
# ✔ FACT_order_details
# ✔ DIM_returned_item
# ✔ DIM_inventory_level
# ✔ DIM_product_forecast
# ✔ FACT_sales_target

In [187]:
satisfaction_type

,SATISFACTION_TYPE_CODE,SATISFACTION_TYPE_DESCRIPTION
0,1,Not satisfied
1,2,Less than satisfied
2,3,Satisfied
3,4,Very Satisfied
4,5,More than satisfied


In [188]:
DIM_Sales_Staff = DIM_Sales_Staff.merge(training, how='left', on='SALES_REPRESENTATIVE_CODE')
DIM_Sales_Staff = DIM_Sales_Staff.rename(columns={'MANAGER_CODE':'MANAGER', 'YEAR': 'TRAINING_YEAR', 'TRAINING_CODE':'TRAINING_COURSE'})
DIM_Sales_Staff = DIM_Sales_Staff.merge(satisfaction, how='left', on='SALES_REPRESENTATIVE_CODE')
DIM_Sales_Staff = DIM_Sales_Staff.merge(satisfaction_type, how='left', on='SATISFACTION_TYPE_CODE')
DIM_Sales_Staff = DIM_Sales_Staff.rename(columns={'YEAR': 'STATISFACTION_YEAR', 'SATISFACTION_TYPE_DESCRIPTION': 'STATISFACTION_DESCRIPTION'})
DIM_Sales_Staff['DATE_HIRED'] = DIM_Sales_Staff['DATE_HIRED'].astype(str)

In [189]:
DIM_Sales_Staff.columns

Index(['SALES_STAFF_CODE', 'SALES_REPRESENTATIVE_CODE', 'FIRST_NAME',
       'LAST_NAME', 'POSITION_EN', 'WORK_PHONE', 'FAX', 'EMAIL', 'DATE_HIRED',
       'SALES_OFFICE_CODE', 'MANAGER', 'EXTENSION', 'SALES_BRANCH_CODE',
       'TRAINING_YEAR', 'COURSE_CODE', 'TRAINING_COURSE', 'STATISFACTION_YEAR',
       'SATISFACTION_TYPE_CODE', 'SATISFACTION_CODE',
       'STATISFACTION_DESCRIPTION'],
      dtype='str')

In [190]:
product = product.merge(product_type, how='left', on='PRODUCT_TYPE_CODE')
product = product.merge(product_line, how='left', on='PRODUCT_LINE_CODE')
product = product.rename(columns={'PRODUCT_IMAGE':'PRODUCT_IMG', 'PRODUCTION_COST':'COST_CATEGORY'})
product['INTRODUCTION_DATE'] = product['INTRODUCTION_DATE'].astype(str)

In [191]:
order_details = order_details.rename(columns={'PRODUCT_NUMBER': 'PRODUCT', 'ORDER_NUMBER': 'ORDER_HEADER'})
customer_headquarters = customer_headquarters.rename(columns={'ADDRESS1' : 'ADRESS1', 'ADDRESS2':'ADRESS2', 'COUNTRY_CODE':'COUNTRY', 'CUSTOMER_CODEMR':'CUSTOMER_CODE_HQ'})
inventory_levels = inventory_levels.rename(columns={'PRODUCT_NUMBER':'PRODUCT'})
product_forecast = product_forecast.rename(columns={'PRODUCT_NUMBER':'PRODUCT'})
sales_target = sales_target.rename(columns={'SALES_STAFF_CODE':'SALES_STAFF', 'PRODUCT_NUMBER':'PRODUCT'})
DIM_Country = DIM_Country.rename(columns={'CURRENCY_NAME':'CURRENCY'})
DIM_Sales_Branch = DIM_Sales_Branch.rename(columns={'COUNTRY_CODE':'COUNTRY', 'ADDRESS1':'ADRESS1', 'ADDRESS2':'ADRESS2', 'POSTAL_ZONE':'POSTAL_CODE'})
DIM_Retailer_Site = DIM_Retailer_Site.rename(columns={'ADDRESS2': 'ADRESS2', 'ADDRESS1':'ADRESS1', 'COUNTRY_CODE':'COUNTRY', 'POSTAL_ZONE':'POSTAL_CODE', 'CUSTOMER_CODE':'CUSTOMER'})
sales_demographic = sales_demographic.rename(columns={'AGE_GROUP_CODE':'AGE_GROUP', 'CUSTOMER_CODEMR':'CUSTOMER_HEADQUARTERS'})
DIM_Sales_Staff = DIM_Sales_Staff.rename(columns={'SALES_BRANCH_CODE':'SALES_BRANCH'})

In [192]:
export_base = sqlite3.connect('dwdb.db')
cur = export_base.cursor()

cur.execute("PRAGMA foreign_keys = OFF;")

# =========================
# 1. ROOT TABLES
# =========================

root = [
    ("DIM_age_group", age_group, [
        "AGE_GROUP_CODE",
        "UPPER_AGE",
        "LOWER_AGE"
    ]),

    ("DIM_customer_segment", customer_segment, [
        "SEGMENT_CODE",
        "LANGUAGE",
        "SEGMENT_NAME",
        "SEGMENT_DESCRIPTION"
    ]),

    ("DIM_Month", Month, [
        "MONTH_ID",
        "MONTHNR",
        "MONTH",
        "QUARTER",
        "YEAR"
    ]),
]

# =========================
# 2. MID LAYER
# =========================

mid = [

    ("DIM_country", DIM_Country, [
        "COUNTRY_CODE",
        "COUNTRY",
        "LANGUAGE",
        "CURRENCY",
        "FLAG_IMAGE",
        "TERRITORY_NAME_EN"
    ]),

    ("DIM_Date", Date, [
        "DATE_ID",
        "DATUM",
        "MONTH"
    ]),

    ("DIM_sales_branch", DIM_Sales_Branch, [
        "SALES_BRANCH_CODE",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_CODE",
        "COUNTRY"
    ]),

    ("DIM_product", product, [
        "PRODUCT_NUMBER",
        "DESCRIPTION",
        "INTRODUCTION_DATE",
        "COST_CATEGORY",
        "MARGIN",
        "PRODUCT_IMG",
        "LANGUAGE",
        "PRODUCT_NAME",
        "PRODUCT_TYPE_EN",
        "PRODUCT_LINE_EN"
    ]),
]

# =========================
# 3. THIRD LAYER
# =========================

third = [

    ("DIM_customer_headquarters", customer_headquarters, [
        "CUSTOMER_CODE_HQ",
        "CUSTOMER_NAME",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_ZONE",
        "PHONE",
        "FAX",
        "COUNTRY",
        "SEGMENT_CODE"
    ]),

    ("DIM_sales_staff", DIM_Sales_Staff, [
        "SALES_STAFF_CODE",
        "FIRST_NAME",
        "LAST_NAME",
        "POSITION_EN",
        "WORK_PHONE",
        "EXTENSION",
        "FAX",
        "DATE_HIRED",
        "TRAINING_YEAR",
        "TRAINING_COURSE",
        "STATISFACTION_YEAR",
        "STATISFACTION_DESCRIPTION",
        "SALES_BRANCH",
        "MANAGER"
    ]),

    ("DIM_customer", customer, [
        "CUSTOMER_CODE",
        "COMPANY_NAME",
        "CUSTOMER_TYPE_EN",
        "CUSTOMER_HEADQUARTERS"
    ]),

    ("DIM_retailer_site", DIM_Retailer_Site, [
        "RETAILER_SITE_CODE",
        "RETAILER_CODE",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_CODE",
        "ACTIVE_INDICATOR",
        "COUNTRY",
        "CUSTOMER"
    ]),
]

# =========================
# 4. FACT TABLES
# =========================

fact = [

    ("FACT_sales_demographic", sales_demographic, [
        "DEMOGRAPHIC_CODE",
        "SALES_PERCENT",
        "CUSTOMER_HEADQUARTERS",
        "AGE_GROUP"
    ]),

    ("FACT_order_header", order_header, [
        "ORDER_NUMBER",
        "RETAILER_NAME",
        "RETAILER_CONTACT_CODE",
        "ORDER_DATE",
        "ORDER_METHOD_EN",
        "RETAILER_SITE",
        "SALES_STAFF",
        "SALES_BRANCH"
    ]),

    ("FACT_order_details", order_details, [
        "ORDER_DETAIL_CODE",
        "QUANTITY",
        "UNIT_COST",
        "UNIT_REVENUE",
        "UNIT_PRICE",
        "UNIT_SALE_PRICE",
        "ORDER_HEADER",
        "PRODUCT"
    ]),

    ("DIM_returned_item", returned_item, [
        "RETURN_CODE",
        "RETURN_DATE",
        "QUANTITY_CATEGORY",
        "RETURN_REASON_EN",
        "ORDER_DETAILS"
    ]),

    ("DIM_inventory_level", inventory_levels, [
        "INVENTORY_LEVELS_CODE",
        "INVENTORY_YEAR",
        "INVENTORY_MONTH",
        "INVENTORY_COUNT",
        "PRODUCT"
    ]),

    ("DIM_product_forecast", product_forecast, [
        "PRODUCT_FORECAST_CODE",
        "YEAR",
        "MONTH",
        "EXPECTED_VOLUME",
        "PRODUCT"
    ]),

    ("FACT_sales_target", sales_target, [
        "SALES_TARGET_CODE",
        "SALES_YEAR",
        "SALES_PERIOD",
        "RETAILER_NAME",
        "RETAILER_CODE",
        "SALES_TARGET",
        "SALES_STAFF",
        "PRODUCT"
    ]),
]

# =========================
# ALL TABLES
# =========================

all_tables = root + mid + third + fact

for table_name, df, cols in all_tables:

    missing = set(cols) - set(df.columns)

    if missing:
        print(f"❌ {table_name} mist kolommen: {missing}")
        continue

    df_clean = df[cols]

    query = f"""
    INSERT OR REPLACE INTO {table_name}
    ({",".join(cols)})
    VALUES ({",".join(["?"] * len(cols))})
    """

    try:
        cur.executemany(query, df_clean.values.tolist())
        print(f"✔ {table_name}")

    except Exception as e:
        print(f"❌ {table_name} ERROR: {e}")
        break

export_base.commit()
export_base.close()

✔ DIM_age_group
✔ DIM_customer_segment
✔ DIM_Month
✔ DIM_country
✔ DIM_Date
✔ DIM_sales_branch
✔ DIM_product
✔ DIM_customer_headquarters
✔ DIM_sales_staff
✔ DIM_customer
✔ DIM_retailer_site
✔ FACT_sales_demographic
✔ FACT_order_header
✔ FACT_order_details
✔ DIM_returned_item
✔ DIM_inventory_level
✔ DIM_product_forecast
✔ FACT_sales_target
